# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing entities by their `@id` as recommended by the Croissant specification.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and initialize Dataset instance
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`, and associated fields. The Croissant `mlcroissant` API provides a list of record sets and fields, each referenced by `@id`.

In [ ]:
# List all record sets with their @id and fields
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    # Display fields @id for record set
    fields = list(getattr(rs, 'fields', []))
    print("  Fields:")
    for field in fields:
        field_id = getattr(field, '@id', '(missing @id)')
        field_name = getattr(field, 'name', '(missing name)')
        print(f"    - {field_id} (name: {field_name})")
    print("\n")

## 3. Data Extraction
This section loads data from each record set using the record set `@id` and constructs a `pandas.DataFrame`. All referencing is done by `@id`.

In [ ]:
# Collect record set @id's discovered above
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  -> Loaded {len(df)} records with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  -> Could not load records for {record_set_id}. Error: {e}")

# Display head of the main tabular dataset (by inspecting DataFrame sizes and fields)
main_record_set_id = max(dataframes, key=lambda rid: dataframes[rid].shape[0] if rid in dataframes else 0)
print(f"\nMain tabular record set selected (by size): {main_record_set_id}")
print("Columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, we'll perform some basic filtering and normalization on a field, referencing columns by their field `@id`. As an example, we'll select the numeric field corresponding to patient age, if available.

_Please refer to the list of fields above to select valid `@id` values for fields and grouping._

In [ ]:
# Select main record set DataFrame for analysis
df = dataframes[main_record_set_id]

# Try common field names for age or similar numeric clinical variable
possible_numeric_fields = [c for c in df.columns if ('age' in c.lower() or 'interval' in c.lower() or df[c].dtype in [int, float, 'int64', 'float64'])]

print(f"Possible numeric fields: {possible_numeric_fields}")

# Choose a numeric field (by @id) for the demo, override this as needed
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0] # Use the first candidate
else:
    raise RuntimeError("No numeric field found in the primary record set.")

# Filtering: Example thresholding on the numeric field
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (threshold = mean):")
print(filtered_df[[numeric_field_id]].head())

# Add normalized version of the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'sex', 'msi' (microsatellite instability status), or similar
possible_group_fields = [c for c in df.columns if 'sex' in c.lower() or 'msi' in c.lower() or df[c].dtype == object]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
    print(grouped_df)
else:
    print("\nNo suitable field for grouping categorical analysis found.")

## 5. Visualization
Let's plot the distribution of the selected numeric field, as well as the normalized scores if available. We also make a grouped bar chart if a categorical grouping was performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

if f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=15, kde=True, color='orange')
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.title(f"Normalized Distribution of {numeric_field_id}")
    plt.show()

# If grouping was performed, make a barplot
if group_field and group_field in filtered_df.columns:
    group_means = filtered_df.groupby(group_field)[numeric_field_id].mean()
    plt.figure(figsize=(6,4))
    sns.barplot(x=group_means.index, y=group_means.values)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load and process the FAIR² dataset, referencing all record sets and fields by their Croissant `@id`. We examined record sets, dynamically discovered field identifiers, and conducted exploratory numeric and categorical analyses. This procedure can be adapted as new field or grouping requirements arise.

**Key findings and next steps:**
- The primary tabular record set contains clinical and pathological data with at least one numeric field suitable for normalization and filtration.
- Filtering and visualizing by demographic or biomarker status (such as MSI) is possible using field `@id` from the metadata overview.
- To expand this analysis, users can select other fields by `@id` (consult cell 5 for candidate field list), or perform more advanced statistical or ML tasks.